In [21]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB ---- collection 단위로 저장
- Pinecone : 클라우드 vector DB ---- index 단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env에 추가(PINECONE_API_KEY등록)


# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


# 1. knowlege base 구성을 위한 데이터 생성

In [22]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수: ', len(documents))

chunk 갯수:  180


In [10]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding= OpenAIEmbeddings(model='text-embedding-3-large')
len(embedding.embed_query('샘플'))

3072

In [23]:
%%time
#pinecone vector database
from pinecone import Pinecone
#from langchain_pinecone import PineconeVectorStore
pc=Pinecone(#api_key=os.getenv('PINECONE_API_KEY')
            )
# 데이터를 처음 저장할 때 
index_name ='tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)


CPU times: total: 46.9 ms
Wall time: 35.4 ms


# 2. 답변 생성을 위한 Retrieval

In [14]:
query="연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3)

In [15]:
retrieved_doc="\n--\n".join([doc.page_content.replace("\n\n","") for doc in retrieved_docs])
#print(retrieved_doc)

In [16]:
retrieved_doc

'제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>1. 「공익신탁법」에 따른 공익신탁의 이익2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.다. 대통령령으로 정하는 농어가부업소득라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 3천만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.바. 대통령령으로 정하는 작물재배업에서 발생하는 소득사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득3. 근로소득과 퇴직소득 중 다음 각 목의 어느 하나에 해당하는 소득가. 대통령령으로 정하는 복무 중인 병(兵)이 받는 

# 3. 답변생성

In [17]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4.1-nano')

In [18]:
from langchain_upstage import ChatUpstage
llm=ChatUpstage(model='solar-pro2')

In [19]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [20]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5,000만 원인 직장인의 소득세를 계산하기 위해서는 **근로소득 공제**, **종합소득공제**, **세율 적용** 등 여러 단계를 거쳐야 합니다. 다음은 2024년 기준 한국 소득세법에 따른 계산 절차입니다.

---

### 1. **근로소득 공제 적용**
- **연봉 5,000만 원**에서 근로소득 공제를 차감합니다.
  - 근로소득 공제율:  
    - 5,000만 원 이하: 15% + 180만 원  
    - 계산: `50,000,000 × 15% + 1,800,000 = 9,300,000원`  
  - **공제 후 금액**: `50,000,000 - 9,300,000 = 40,700,000원`

---

### 2. **종합소득공제 적용**
- **기본공제**: 본인 1,500,000원 (추가 부양가족 없을 경우)  
- **연금보험료 공제**: 국민연금 납부액 (예: 월 20만 원 × 12 = 2,400,000원)  
- **보험료 공제**: 건강보험·고용보험 등 (예: 2,000,000원)  
- **의료비·교육비 공제**: 해당 금액 추가 가능  

**예시 공제액**:  
`1,500,000 (기본) + 2,400,000 (연금) + 2,000,000 (보험료) = 5,900,000원`  
- **공제 후 과세표준**: `40,700,000 - 5,900,000 = 34,800,000원`

> ※ 실제로는 더 많은 공제 항목(주택자금, 신용카드 등)이 적용될 수 있으나, 여기서는 단순화하여 계산했습니다.

---

### 3. **세율 적용 (종합소득세)**
- 2024년 종합소득세 세율은 다음과 같습니다:  
  - 1,200만 원 이하: 6%  
  - 1,200~4,600만 원: 15% (누적 1,080,000원)  
  - 4,600~8,800만 원: 24% (누적 5,220,000원)  
  - ... (이하 생략)  

**과세표준 34,800,000원**에 대한 세액 계산:  
- 1,200만 원 × 6% = 720,00

# 4. langchain으로 답변 생성

In [15]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
                     ('system', '당신은 최고의 한국 소득세법 전문가입니다'),
                     ('human', f'''다음 문맥을 참고하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : ''')
                    ]) 

In [16]:
prompt = promptTemplate.invoke({'context':retrieved_doc, 
                                'question':query})

In [17]:
#과금
ai_message = llm.invoke(prompt)

In [18]:
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'한국의 소득세는 납세자의 과세표준과 세율에 따라 차등 부과됩니다. 연봉 5천만 원인 직장인은 근로소득에 대해 일정 공제 후 세율을 적용받으며, 대략적인 세금액은 약 800만 원~1,500만 원 수준입니다. 정확한 세액은 공제 항목, 기타 소득 또는 공제 대상이 되는 소득공제 등을 고려하여 산출해야 합니다.'

In [19]:
# 위의 작업을 한번에
outputParser.invoke(llm.invoke(promptTemplate.invoke(
    {'context':retrieved_doc,'question':query}
)))

'연봉 5천만원인 직장인이 납부해야 할 소득세는 여러 공제와 세율에 따라 달라지나, 일반적으로 근로소득기본공제와 표준세율을 적용하면 약 1,000만 원 내외의 세액이 예상됩니다. 정확한 세액 계산을 위해서는 공제사항 및 기타 소득 내용을 고려해야 합니다.'

In [20]:
rag_chain =promptTemplate | llm | outputParser
rag_chain.invoke({'context':retrieved_doc, 'question': query})


'연봉 5천만원인 근로자는 기본공제, 특별공제, 인적공제 등을 고려한 후 과세표준에 따라 소득세를 납부하게 됩니다. 2023년 기준으로, 연간 소득세는 대략 1,000만 원 내외로 예상됩니다. 정확한 세액은 공제 사항과 세율에 따라 달라질 수 있으므로 구체적인 계산이 필요합니다.'

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import ChatUpstage
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
llm = ChatUpstage(model = 'solar-pro4')
# 2. vector DB load
pc = Pinecone()
index_name = 'tax-index-openai'
vectorStore = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)